<a href="https://colab.research.google.com/github/AarnavSawant/KVCompass/blob/main/KVCompass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KVCompass Team Demo Notebook

This notebook is organized so Tony, Will, and Grady can run disjoint sweeps in separate Colab sessions and combine the results for Friday. Use the shared setup cells first, then only run the teammate section assigned to you, then use the final aggregation cells for the presentation.


## Team Split

- Tony: baseline and main balanced methods on retrieval-long
- Will: additional retrieval methods on retrieval-long
- Grady: aggressive-memory methods on memory-tight

Everyone should use the same model and branch, and each person should only run their assigned sweep cell.


In [ ]:
# Shared setup: clone or refresh the repo, switch to the demo branch, and install.
from pathlib import Path

repo_dir = Path('/content/KVCompass')
if not repo_dir.exists():
    !git clone https://github.com/AarnavSawant/KVCompass.git /content/KVCompass

%cd /content/KVCompass
!git fetch origin
!git checkout codex/colab-sweep-workflow
!git pull
!nvidia-smi
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .


In [ ]:
# Shared auth: set HF_TOKEN from Colab secrets and verify login.
from google.colab import userdata
from huggingface_hub import whoami
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print(whoami())


In [ ]:
# Shared helper: write a sweep YAML string to disk.
from pathlib import Path

def write_sweep(name: str, yaml_text: str) -> Path:
    path = Path(f"configs/{name}.yaml")
    path.write_text(yaml_text, encoding="utf-8")
    print(path.read_text())
    return path


## Tony Runs

Run the next two Tony cells only.


In [ ]:
# Tony: baseline + core balanced methods on retrieval-long.
tony_config = write_sweep("benchmark_sweeps.tony", """
sweep:
  name: tony_retrieval_core
  model: Qwen/Qwen2.5-1.5B-Instruct
  device: auto
  torch_dtype: bfloat16
  methods_config_path: configs/methods.yaml
  output_dir: results/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: retrieval_long
      dataset: ruler
      data_dir: "4096"
      fraction: 0.02
      methods:
        - no_compression
        - snapkv
        - expected_attention
      budgets:
        no_compression: [1.0]
        default: [0.5]
""")


In [ ]:
# Tony run command.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.tony.yaml


## Will Runs

Run the next two Will cells only.


In [ ]:
# Will: additional retrieval methods on retrieval-long.
will_config = write_sweep("benchmark_sweeps.will", """
sweep:
  name: will_retrieval_extended
  model: Qwen/Qwen2.5-1.5B-Instruct
  device: auto
  torch_dtype: bfloat16
  methods_config_path: configs/methods.yaml
  output_dir: results/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: retrieval_long
      dataset: ruler
      data_dir: "4096"
      fraction: 0.02
      methods:
        - adakv_expected
        - pyramidkv
        - streaming_llm
      budgets:
        default: [0.5]
""")


In [ ]:
# Will run command.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.will.yaml


## Grady Runs

Run the next two Grady cells only.


In [ ]:
# Grady: aggressive-memory methods on memory-tight.
grady_config = write_sweep("benchmark_sweeps.grady", """
sweep:
  name: grady_memory_tight
  model: Qwen/Qwen2.5-1.5B-Instruct
  device: auto
  torch_dtype: bfloat16
  methods_config_path: configs/methods.yaml
  output_dir: results/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: memory_tight
      dataset: ruler
      data_dir: "4096"
      fraction: 0.02
      methods:
        - knorm
        - tova
        - random
      budgets:
        default: [0.25]
""")


In [ ]:
# Grady run command.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.grady.yaml


## Final Aggregation For The Demo

After all three teammates finish and sync results, run these cells to build the presentation tables and plots.


In [ ]:
# Load all teammate summary CSVs and build one combined leaderboard.
import json
from pathlib import Path
import pandas as pd

summary_paths = sorted(Path("results/benchmark_eval").glob("*__summary.csv"))
display(pd.DataFrame({"summary_csv": [str(p) for p in summary_paths]}))

summary_df = pd.concat([pd.read_csv(path) for path in summary_paths], ignore_index=True)
display(summary_df)

rows = []
for _, row in summary_df.iterrows():
    metrics = json.loads(Path(row["metrics_path"]).read_text())
    metric_values = []
    for task_metrics in metrics.values():
        if isinstance(task_metrics, dict):
            metric_values.extend(v for v in task_metrics.values() if isinstance(v, (int, float)))
    rows.append({
        "method": row["method"],
        "budget": row["budget"],
        "avg_quality": round(sum(metric_values) / len(metric_values), 2) if metric_values else None,
        "avg_latency_seconds": row.get("avg_latency_seconds"),
        "avg_throughput_tokens_per_second": row.get("avg_throughput_tokens_per_second"),
        "peak_gpu_memory_mb": row.get("peak_gpu_memory_mb"),
        "dataset": row.get("dataset"),
    })

leaderboard_df = pd.DataFrame(rows).sort_values(["avg_quality", "avg_latency_seconds"], ascending=[False, True])
print("Combined Demo Leaderboard")
display(leaderboard_df)


In [ ]:
# Plot the main tradeoffs for the presentation.
import matplotlib.pyplot as plt

plot_df = leaderboard_df.copy()
plot_df["label"] = plot_df["method"] + " @ " + plot_df["budget"].astype(str)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].bar(plot_df["label"], plot_df["avg_quality"], color="#2a6f97")
axes[0].set_title("Average Benchmark Quality")
axes[0].tick_params(axis="x", rotation=60)

axes[1].bar(plot_df["label"], plot_df["avg_latency_seconds"], color="#c1121f")
axes[1].set_title("Average Latency (s)")
axes[1].tick_params(axis="x", rotation=60)

axes[2].bar(plot_df["label"], plot_df["peak_gpu_memory_mb"], color="#6a994e")
axes[2].set_title("Peak GPU Memory (MB)")
axes[2].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()


In [ ]:
# Build a simple recommendation table for the presentation.
best_quality = leaderboard_df.sort_values(["avg_quality", "avg_latency_seconds"], ascending=[False, True]).iloc[0]
best_latency = leaderboard_df.sort_values("avg_latency_seconds", ascending=True).iloc[0]
best_memory = leaderboard_df.dropna(subset=["peak_gpu_memory_mb"]).sort_values("peak_gpu_memory_mb", ascending=True).iloc[0] if leaderboard_df["peak_gpu_memory_mb"].notna().any() else None
balanced_df = leaderboard_df.copy()
for col in ["avg_quality", "avg_latency_seconds"]:
    vals = balanced_df[col].astype(float)
    if col == "avg_quality":
        balanced_df[col + "_norm"] = (vals - vals.min()) / (vals.max() - vals.min() + 1e-8)
    else:
        balanced_df[col + "_norm"] = 1 - ((vals - vals.min()) / (vals.max() - vals.min() + 1e-8))
balanced_df["balanced_score"] = 0.6 * balanced_df["avg_quality_norm"] + 0.4 * balanced_df["avg_latency_seconds_norm"]
best_balanced = balanced_df.sort_values("balanced_score", ascending=False).iloc[0]

rec_rows = [
    {"category": "best_for_quality", "method": best_quality["method"], "budget": best_quality["budget"]},
    {"category": "best_for_latency", "method": best_latency["method"], "budget": best_latency["budget"]},
]
if best_memory is not None:
    rec_rows.append({"category": "best_for_memory", "method": best_memory["method"], "budget": best_memory["budget"]})
rec_rows.append({"category": "best_balanced", "method": best_balanced["method"], "budget": best_balanced["budget"]})
recommendation_df = pd.DataFrame(rec_rows)
display(recommendation_df)


In [ ]:
# Optional smoke test: one tiny direct benchmark run.
!python scripts/run_kvpress_benchmark_eval.py \
  --dataset ruler \
  --data-dir 4096 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --method no_compression \
  --budget 1.0 \
  --fraction 0.002 \
  --torch-dtype bfloat16
